In [1]:
!pip install pypdf

In [2]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 8.4 MB/s  0:00:02
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-convent

In [1]:
#Import libraries

import chromadb
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


#Upload PDF

uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_name)


#Extract text from PDF page by page

reader = PdfReader(pdf_name)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text.strip():
        pages.append({
            "page": page_number,
            "text": text
        })

print("Total Pages Extracted:", len(pages))


# Split text into chunks

def chunk_text(text, chunk_size=500, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# Create chunks and preserve page numbers

all_chunks = []
all_metadata = []

for page in pages:

    page_chunks = chunk_text(
        page["text"],
        chunk_size=500,
        overlap=100
    )

    for chunk_number, chunk in enumerate(page_chunks):

        all_chunks.append(chunk)

        all_metadata.append({
            "source": pdf_name,
            "page": page["page"],
            "chunk_number": chunk_number + 1
        })


print("Total Chunks Created:", len(all_chunks))

Saving Dwise_Machine_Learning_Training_Course_Distribution.pdf to Dwise_Machine_Learning_Training_Course_Distribution.pdf
Uploaded PDF: Dwise_Machine_Learning_Training_Course_Distribution.pdf
Total Pages Extracted: 10
Total Chunks Created: 42


In [2]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
document_embeddings = model.encode(all_chunks)

In [4]:
print(document_embeddings.shape)

(42, 384)


In [5]:
client = chromadb.Client()

In [6]:
collection = client.create_collection(
    name = "pdf_semantic_search",
    embedding_function =None,
    metadata = {"hnsw:space": "cosine"}
)

In [7]:
ids = [
    f"chunk_{i}" for i in range(len(all_chunks))
]

In [8]:
print(ids)

['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4', 'chunk_5', 'chunk_6', 'chunk_7', 'chunk_8', 'chunk_9', 'chunk_10', 'chunk_11', 'chunk_12', 'chunk_13', 'chunk_14', 'chunk_15', 'chunk_16', 'chunk_17', 'chunk_18', 'chunk_19', 'chunk_20', 'chunk_21', 'chunk_22', 'chunk_23', 'chunk_24', 'chunk_25', 'chunk_26', 'chunk_27', 'chunk_28', 'chunk_29', 'chunk_30', 'chunk_31', 'chunk_32', 'chunk_33', 'chunk_34', 'chunk_35', 'chunk_36', 'chunk_37', 'chunk_38', 'chunk_39', 'chunk_40', 'chunk_41']


In [9]:
collection.add(
    ids = ids,
    documents = all_chunks,
    metadatas = all_metadata,
    embeddings = document_embeddings.tolist()
)

In [10]:
def semantic_search(query):
  query_embedding = model.encode([query])

  results = collection.query(
      query_embeddings = query_embedding.tolist(),
      n_results = 2
  )

  for i, document in enumerate(results["documents"][0]):
    print(f"result {i+1} ")
    print("Document:", document)

    print("Source:", results["metadatas"][0][i]["source"])

    print("Page:", results["metadatas"][0][i]["page"])

    print("Distance:", round(results["distances"][0][i], 4))


In [11]:
while True:
  query = input("Enter your query :")
  if query.lower() == "exit":
    break
  semantic_search(query)

Enter your query :histogram
result 1 
Document: ionships, 
distributions and possible issues in the dataset through visualizations. 
Univariate Analysis 
 Numerical data — Histogram 
 Categorical data — Count plot 
Bivariate Analysis 
 Numerical vs Numerical 
 Categorical vs Numerical 
 Scatter plot 
 Box plot 
Multivariate Analysis 
 Correlation Heatmap 
By the end of Day 2, students understand how to perform basic as well as advanced EDA on a regression dataset. 
DAY 3 — First Half 
Preprocessing + Regression Models + Evaluation + St
Source: Dwise_Machine_Learning_Training_Course_Distribution.pdf
Page: 2
Distance: 0.6086
result 2 
Document: ut variables 
 Why the problem is a regression problem 
2. Basic EDA 
 Dataset shape 
 Number of rows and columns 
 Column names 
 Dataset information 
 Data types 
 Statistical summary 
 Missing values 
 Numerical and categorical variables 
 Basic distribution 
 Skewness 
3. Advanced EDA 
After basic EDA, students move towards v